# Rotina diária – Daily Experiência BR no Databricks

Rode as células em ordem. Cada passo pode ser repetido sem problema: o envio só manda arquivos novos ou alterados e
nunca apaga nada; o job só pode ter uma execução por vez (uma segunda fica na fila).

1. **Google Sheets** – confere se a exportação de hoje (Apps Script, por volta das 9h) já chegou ao `G:\`.
2. **Envio** – manda ao volume os arquivos novos: brutos do MKTZap, planilhas do Drive e Google Sheets.
3. **Job** – dispara o processamento no Databricks (bronze → silver → gold, uns 15 minutos) e acompanha.
4. **Conferência** – mostra as tabelas gold e quando foram atualizadas.

In [ ]:
from datetime import datetime

import enviar_arquivos as rotina

cfg = rotina.carregar_config()
w = rotina.cliente(cfg)
print("Conectado ao Databricks como", w.current_user.me().user_name)

## 1. As Google Sheets de hoje já foram exportadas?

In [ ]:
pronta, mensagem = rotina.verificar_google_sheets(cfg)
print(mensagem)

## 2. Envio dos arquivos

Se a célula anterior ainda não mostrou ✅, espere alguns minutos e rode-a de novo. Para ver o que seria enviado sem
enviar, use `rotina.enviar(cfg, simular=True)`.

In [ ]:
resultados = rotina.enviar(cfg, w)
contagem = rotina.imprimir_resultados(cfg, resultados)

## 3. Processamento no Databricks

In [ ]:
if contagem["erro"]:
    raise RuntimeError("Houve erro no envio (veja a célula anterior); o job não foi disparado.")
run_id = rotina.disparar_job(w, cfg["destino"]["job"])

In [ ]:
# Mostra o andamento de cada tarefa a cada 30 segundos até o job terminar (pode interromper e rodar de novo depois)
rotina.acompanhar_job(w, run_id)

## 4. Conferência das tabelas gold

In [ ]:
import pandas as pd

pd.DataFrame(rotina.conferir_gold(w, cfg), columns=["tabela", "linhas", "última gravação"])